This is the notebook for experimention different chunking strategy and vector embedding to test RAG pipeline.

This notebook is reference from an online course on RAG implementation by VIZUARA: Vizuara is an AI training platform.

In [1]:
#!pip install PyMuPDF tqdm sentence-transformers accelerate bitsandbytes flash-attn --no-build-isolation

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
BASE_DIR = "/content/drive/MyDrive/10K_PDFs"


In [16]:
## To show the list of files

from pathlib import Path

p = Path(BASE_DIR)
pdf_files = sorted(p.glob("**/*.pdf"))
csv_files = sorted(p.glob("**/*.csv"))

print(f"PDFs: {len(pdf_files)} files")
print(f"CSVs: {len(csv_files)} files")

pdf_files[:5], csv_files[:5]


PDFs: 32 files
CSVs: 1 files


([PosixPath('/content/drive/MyDrive/10K_PDFs/3M_10K.pdf'),
  PosixPath('/content/drive/MyDrive/10K_PDFs/AIG_10K.pdf'),
  PosixPath('/content/drive/MyDrive/10K_PDFs/AT&T_10K.pdf'),
  PosixPath('/content/drive/MyDrive/10K_PDFs/Abbott_Laboratories_10K.pdf'),
  PosixPath('/content/drive/MyDrive/10K_PDFs/Alphabet_10K.pdf')],
 [PosixPath('/content/drive/MyDrive/10K_PDFs/yahoo_most_active_stocks.csv')])

In [5]:
import re
import fitz
import hashlib
from pathlib import Path
from typing import List, Dict

def text_processor(text: str) -> str:
    """
    Text pre processing and cleaning before storing it for chunking
    """
    if not text:
        return ""
    t = text.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", t)
    lines = [re.sub(r"[ \t]+", " ", ln.strip()) for ln in t.split("\n")]
    t = " ".join([ln for ln in lines if ln])
    t = re.sub(r"\s+", " ", t).strip()
    return t


def read_pdf(pdf_path: str) -> List[Dict]:
    """
    Read a PDF and return a list of page dicts ||||||| The list of all the dictionaries should be formatted as per below data
      doc_id,
      filename,
      page_no,
      char_count,
      word_count,
      sentence_count,
      page_token_count,
      text_clean
    """
    path = Path(pdf_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    st = path.stat()
    doc_id = hashlib.md5(f"{path.resolve()}|{st.st_size}|{int(st.st_mtime)}".encode("utf-8")).hexdigest()

    records: List[Dict] = []
    with fitz.open(str(path)) as doc:
        for i in range(len(doc)):
            page = doc.load_page(i)
            raw = page.get_text("text") or ""
            clean = text_processor(raw)

            words = clean.split()
            word_count = len(words)

            sentences = [s for s in re.split(r"(?<=[.!?])\s+", clean) if s.strip()] if clean else []
            sentence_count = len(sentences)

            records.append({
                "doc_id": doc_id,
                "filename": path.name,
                "page_no": i + 1,
                "char_count": len(clean),
                "word_count": word_count,
                "sentence_count": sentence_count,
                "page_token_count": word_count,
                "text_clean": clean,
            })
    return records


In [6]:
from pathlib import Path
from typing import List, Dict

def read_pdfs(pdf_paths: List[str]) -> List[Dict]:
    """
    This will process all my pdfs at a time
    This code is referenced from Gemini
    """
    all_records: List[Dict] = []
    for p in pdf_paths:
        try:
            all_records.extend(read_pdf(p))
        except Exception as e:
            print(f"[skip] {p}: {e}")
    return all_records


def read_pdfs_in_dir(folder: str, recursive: bool = True) -> List[Dict]:
    """
    extract all the pdf and merge in the list
    """
    base = Path(folder)
    if not base.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    pdf_iter = base.rglob("*.pdf") if recursive else base.glob("*.pdf")
    pdf_paths = [str(p) for p in pdf_iter]

    # ignore CSVs or others file format
    pdf_paths = [p for p in pdf_paths if p.lower().endswith(".pdf")]

    return read_pdfs(pdf_paths)


In [19]:
# Process all the pdfs under this folder
all_pages = read_pdfs_in_dir("/content/drive/MyDrive/10K_PDFs", recursive=True)
# Corss check
len(all_pages), all_pages[100]


(3943,
 {'doc_id': '606f248247422e3fb28b2bab06659955',
  'filename': 'AIG_10K.pdf',
  'page_no': 41,
  'char_count': 490,
  'word_count': 77,
  'sentence_count': 4,
  'page_token_count': 77,
  'text_clean': 'Table of Contents SIGNATURES Pursuant to the requirements of the Securities Exchange Act of 1934, the registrant has duly caused this report to be signed on its behalf by the undersigned thereunto duly authorized. ALPHABET INC. Date: March 29, 2016 By: /s/ LARRY PAGE Larry Page Chief Executive Officer (Principal Executive Officer of Alphabet Inc.) GOOGLE INC. Date: March 29, 2016 By: /s/ SUNDAR PICHAI Sundar Pichai Chief Executive Officer (Principal Executive Officer of Google Inc.) 37'})

In [23]:
all_pages[35]

{'doc_id': 'c340c0cb912d4aad0efa9e652c9eeb63',
 'filename': 'apple_2024_10K.pdf',
 'page_no': 36,
 'char_count': 2228,
 'word_count': 305,
 'sentence_count': 4,
 'page_token_count': 305,
 'text_clean': 'Apple Inc. CONSOLIDATED STATEMENTS OF CASH FLOWS (In millions) Years ended September 28, 2024 September 30, 2023 September 24, 2022 Cash, cash equivalents, and restricted cash and cash equivalents, beginning balances $ 30,737 $ 24,977 $ 35,929 Operating activities: Net income 93,736 96,995 99,803 Adjustments to reconcile net income to cash generated by operating activities: Depreciation and amortization 11,445 11,519 11,104 Share-based compensation expense 11,688 10,833 9,038 Other (2,266) (2,227) 1,006 Changes in operating assets and liabilities: Accounts receivable, net (3,788) (1,688) (1,823) Vendor non-trade receivables (1,356) 1,271 (7,520) Inventories (1,046) (1,618) 1,484 Other current and non-current assets (11,731) (5,684) (6,499) Accounts payable 6,020 (1,889) 9,448 Other curr

In [27]:
all_pages[167]['text_clean']

"Redeemable Noncontrolling Interest Noncontrolling interests that are redeemable outside the Company's control at fixed or determinable prices and dates are presented as temporary equity in the Consolidated Balance Sheets. Redeemable noncontrolling interests are recorded at the greater of the redemption fair value or the carrying value of the noncontrolling interest and adjusted each reporting period for income, loss and any distributions made. Remeasurements to the redemption value of the redeemable noncontrolling interest are recognized in capital in excess of par. The Company has a redeemable noncontrolling interest related to an acquisition in the Walmart U.S. segment as the minority interest owner holds a put option which may require the Company to purchase its interest beginning in December 2027, with annual options thereafter. Revenue Recognition Net Sales The Company recognizes sales revenue, net of sales taxes and estimated sales returns, at the time it sells merchandise or pr

In [17]:
import os, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

out_dir = "/content/drive/MyDrive/10k_processed"
os.makedirs(out_dir, exist_ok=True)

df = pd.DataFrame(all_pages)
df.to_parquet(f"{out_dir}/pages.parquet", engine="pyarrow", compression="zstd", index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd

# for understanding our data
df = pd.DataFrame(all_pages)

df.head(5)

,doc_id,filename,page_no,char_count,word_count,sentence_count,page_token_count,text_clean
0,c340c0cb912d4aad0efa9e652c9eeb63,apple_2024_10K.pdf,1,2063,354,8,354,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
1,c340c0cb912d4aad0efa9e652c9eeb63,apple_2024_10K.pdf,2,3123,481,16,481,Indicate by check mark whether the Registrant ...
2,c340c0cb912d4aad0efa9e652c9eeb63,apple_2024_10K.pdf,3,1350,208,25,208,Apple Inc. Form 10-K For the Fiscal Year Ended...
3,c340c0cb912d4aad0efa9e652c9eeb63,apple_2024_10K.pdf,4,3628,575,26,575,This Annual Report on Form 10-K (“Form 10-K”) ...
4,c340c0cb912d4aad0efa9e652c9eeb63,apple_2024_10K.pdf,5,3648,530,22,530,Services Advertising The Company’s advertising...


In [15]:
df.describe()

,page_no,char_count,word_count,sentence_count,page_token_count
count,3943.000000,3943.000000,3943.000000,3943.000000,3943.000000
mean,97.208978,3415.208978,517.459802,17.947248,517.459802
std,107.557748,1916.907913,283.012394,12.331966,283.012394
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,32.000000,2017.000000,315.000000,8.000000,315.000000
50%,68.000000,3526.000000,541.000000,18.000000,541.000000
75%,112.000000,4898.000000,734.000000,27.000000,734.000000
max,569.000000,9150.000000,1362.000000,78.000000,1362.000000


## Chunking

In [30]:
## First we will clean the pages where total word is less than 50
from typing import List, Dict, Tuple

def drop_junk_pages(records: List[Dict], min_words: int = 50) -> Tuple[List[Dict], List[Dict]]:
    """
    Keep pages with word_count >= min_words
    This is to exclude unncessary information from our dataset
    """
    get_wc = lambda r: int(r.get("word_count") or 0)
    kept   = [r for r in records if get_wc(r) >= min_words]
    junk   = [r for r in records if get_wc(r) <  min_words]
    return kept, junk



kept_pages, junk_pages = drop_junk_pages(all_pages, min_words=50)
len(kept_pages), len(junk_pages)



(3603, 340)

In [31]:
import re
from typing import List, Dict
## -------------------------------------------------
## sfixed size splitter
## Splitting 10 sentences with 2 overlap sentences
## This code is referencced from cahtgpt
## -------------------------------------------------
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def split_sentences_with_pages(pages: List[Dict]) -> List[tuple]:
    """pages: list of page dicts for one doc (sorted by page_no).
       returns [(sentence, page_no), ...]"""
    out = []
    for p in pages:
        txt = (p.get("text_clean") or "").strip()
        if not txt:
            continue
        parts = [s.strip() for s in SENT_SPLIT.split(txt) if s.strip()]
        out.extend((s, int(p["page_no"])) for s in parts)
    return out

def chunk_by_sentences(
    records: List[Dict],
    sentences_per_chunk: int = 10,
    overlap: int = 2
) -> List[Dict]:
    """Fixed-size sentence chunking per document (doc_id)."""
    assert sentences_per_chunk > 0 and 0 <= overlap < sentences_per_chunk
    step = sentences_per_chunk - overlap

    # Specific to individual documents
    by_doc: Dict[str, List[Dict]] = {}
    for r in records:
        by_doc.setdefault(r["doc_id"], []).append(r)

    chunks: List[Dict] = []

    for doc_id, pages in by_doc.items():
        pages = sorted(pages, key=lambda r: int(r["page_no"]))
        filename = pages[0]["filename"] if pages else "unknown.pdf"

        sent_with_pages = split_sentences_with_pages(pages)
        n = len(sent_with_pages)
        if n == 0:
            continue

        chunk_id = 1
        i = 0
        while i < n:
            j = min(i + sentences_per_chunk, n)
            sentences = [s for s, _pg in sent_with_pages[i:j]]
            src_pages = [pg for _s, pg in sent_with_pages[i:j]]

            text = " ".join(sentences).strip()
            words = text.split()
            chunk = {
                "doc_id": doc_id,
                "filename": filename,
                "chunk_id": chunk_id,
                "page_start": min(src_pages),
                "page_end": max(src_pages),
                "source_pages": sorted(set(src_pages)),
                "sentence_count": len(sentences),
                "word_count": len(words),
                "char_count": len(text),
                "page_token_count": len(words),
                "text_clean": text,
            }
            chunks.append(chunk)

            chunk_id += 1
            if j == n:
                break
            i += step


    chunks.sort(key=lambda r: (r["filename"], r["chunk_id"]))
    return chunks


In [32]:
## initialising the chunking based on 10 snetence and 2 overlap sentence
chunks = chunk_by_sentences(kept_pages, sentences_per_chunk=10, overlap=2)
len(chunks), chunks[0]


(8817,
 {'doc_id': '9cf140658edcf90c20d3825c90fd9c0f',
  'filename': '3M_10K.pdf',
  'chunk_id': 1,
  'page_start': 1,
  'page_end': 1,
  'source_pages': [1],
  'sentence_count': 10,
  'word_count': 348,
  'char_count': 2076,
  'page_token_count': 348,
  'text_clean': 'Table of Contents UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2024 or o TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from __________ to __________ Commission file number 1-3285 3M COMPANY State of Incorporation: Delaware I.R.S. Employer Identification No. 41-0417775 Principal executive offices: 3M Center, St. Paul, Minnesota 55144 Telephone number: (651) 733-1110 Securities registered pursuant to Section 12(b) of the Act: Title of each class Trading Symbol(s) Name of each exchange on which r

In [45]:
len(chunks), chunks[346]

(8817,
 {'doc_id': '606f248247422e3fb28b2bab06659955',
  'filename': 'AIG_10K.pdf',
  'chunk_id': 14,
  'page_start': 6,
  'page_end': 6,
  'source_pages': [6],
  'sentence_count': 10,
  'word_count': 289,
  'char_count': 1848,
  'page_token_count': 289,
  'text_clean': 'John holds a Master of Business Administration degree from Harvard Business School, and a Master of Science degree in electrical engineering and computer science, and a Bachelor of Science degree in electrical engineering from Rice University. Diane B. Greene has served as a member of our Board of Directors since January 2012 and as a Senior Vice President of Google since December 2015. Diane has also been a member of the board of directors of Intuit Inc., a provider of business and financial management solutions, since August 2006 and serves on its audit and risk committee and nominating and corporate governance committee. Diane co-founded VMware, Inc., a virtualization software company, in 1998 and took the company p

In [46]:
len(chunks), chunks[347]

(8817,
 {'doc_id': '606f248247422e3fb28b2bab06659955',
  'filename': 'AIG_10K.pdf',
  'chunk_id': 15,
  'page_start': 6,
  'page_end': 6,
  'source_pages': [6],
  'sentence_count': 10,
  'word_count': 226,
  'char_count': 1398,
  'page_token_count': 226,
  'text_clean': 'Diane holds a Master of Science degree in computer science from the University of California, Berkeley, a Master of Science degree in naval architecture from the Massachusetts Institute of Technology, and a Bachelor of Arts degree in mechanical engineering from the University of Vermont. John L. Hennessy has served as a member of our Board of Directors since April 2004, and as Lead Independent Director since April 2007. John has served as the President of Stanford University since September 2000. John has also been a member of the board of directors of Cisco Systems, Inc., a networking equipment company, since January 2002, and serves on its nominating and governance committee and acquisition committee. He also serves 

In [47]:
len(chunks), chunks[348]

(8817,
 {'doc_id': '606f248247422e3fb28b2bab06659955',
  'filename': 'AIG_10K.pdf',
  'chunk_id': 16,
  'page_start': 6,
  'page_end': 6,
  'source_pages': [6],
  'sentence_count': 10,
  'word_count': 250,
  'char_count': 1514,
  'page_token_count': 250,
  'text_clean': 'John has announced that he plans to resign from his position as the President of Stanford University in August 2016. Ann Mather has served as a member of our Board of Directors since November 2005. Ann has also been a member of the board of directors of: Arista Networks, Inc., a computer networking company, since June 2013, and serves as chair of its audit committee; Glu Mobile Inc., a publisher of mobile games, since September 2005, and serves on its nominating and corporate governance committee; Netflix, Inc., a streaming media company, since July 2010, and serves as chair of its audit committee; and Shutterfly, Inc., an internet-based image publishing company, since May 2013 and serves on its audit committee. Ann ha

In [48]:
len(chunks), chunks[349]

(8817,
 {'doc_id': '606f248247422e3fb28b2bab06659955',
  'filename': 'AIG_10K.pdf',
  'chunk_id': 17,
  'page_start': 6,
  'page_end': 6,
  'source_pages': [6],
  'sentence_count': 10,
  'word_count': 158,
  'char_count': 958,
  'page_token_count': 158,
  'text_clean': 'Alan R. Mulally has served as a member of our Board of Directors since July 2014. Alan served as President and Chief Executive Officer of Ford Motor Company, a global automotive company, from September 2006 through June 2014. Alan was previously a member of the board of directors of Ford and served on its finance committee from September 2006 through June 2014. From March 2001 to September 2006, Alan served as Executive Vice President of the Boeing Company and President and Chief Executive Officer of Boeing Commercial Airplanes, Inc. He also was a member of the Boeing Executive Council. Prior to that time, he served as President of Boeing’s space and defense business. Alan served as co-chair of the Washington Competitiv

Embedding using sentence-transformers/all-mpnet-base-v2

In [51]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="sentence-transformers/all-mpnet-base-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [52]:
sentences = [
    """Alan R. Mulally has served as a member of our Board of Directors since July 2014.
    Alan served as President and Chief Executive Officer of Ford Motor Company, a global automotive company,
    from September 2006 through June 2014. Alan was previously a member of the board of directors of Ford and served on its finance committee
    from September 2006 through June 2014. From March 2001 to September 2006, Alan served as Executive Vice President of the Boeing Company
    and President and Chief Executive Officer of Boeing Commercial Airplanes, Inc. He also was a member of the Boeing Executive Council.
    Prior to that time, he served as President of Boeing’s space and defense business.
    Alan served as co-chair of the Washington Competitiveness Council and sat on the advisory boards of NASA,
    the University of Washington, the University of Kansas, the Massachusetts Institute of Technology,
    and the U.S. Air Force Scientific Advisory Board. He is a member of the U.S.
    """
]

In [53]:
embeddings = embedding_model.encode(sentences)
embedding_dict = dict(zip(sentences, embeddings))

for sentence, embedding in embedding_dict.items():
  print("Sentence :", sentence)
  print("Embedding :", embedding)
  print("")

Sentence : Alan R. Mulally has served as a member of our Board of Directors since July 2014. 
    Alan served as President and Chief Executive Officer of Ford Motor Company, a global automotive company, 
    from September 2006 through June 2014. Alan was previously a member of the board of directors of Ford and served on its finance committee 
    from September 2006 through June 2014. From March 2001 to September 2006, Alan served as Executive Vice President of the Boeing Company 
    and President and Chief Executive Officer of Boeing Commercial Airplanes, Inc. He also was a member of the Boeing Executive Council. 
    Prior to that time, he served as President of Boeing’s space and defense business. 
    Alan served as co-chair of the Washington Competitiveness Council and sat on the advisory boards of NASA, 
    the University of Washington, the University of Kansas, the Massachusetts Institute of Technology, 
    and the U.S. Air Force Scientific Advisory Board. He is a member of

In [57]:
## Vector embedding my fixed chunks

embedding_model.max_seq_length = 384  ## mpnet can process max 384 words for embedding
texts = [c["text_clean"] for c in chunks]
emb = embedding_model.encode(
    texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
emb.shape


Batches:   0%|          | 0/138 [00:00<?, ?it/s]

(8817, 768)

In [59]:
import pandas as pd
df_meta = pd.DataFrame(chunks)
df_meta["embedding_dim"] = emb.shape[1]

df_meta["embedding"] = [v.astype(float).tolist() for v in emb]

row0 = df_meta.iloc[467].to_dict()
import json; print(json.dumps(row0, indent=2, ensure_ascii=False))


{
  "doc_id": "4b0038a49b90617ab24f1a1d0f1f697b",
  "filename": "AT&T_10K.pdf",
  "chunk_id": 14,
  "page_start": 4,
  "page_end": 5,
  "source_pages": [
    4,
    5
  ],
  "sentence_count": 10,
  "word_count": 181,
  "char_count": 1308,
  "page_token_count": 181,
  "text_clean": "1 AT&T Inc. Dollars in millions except per share amounts General We are a leading provider of telecommunications and technology services globally. The services and products that we offer vary by market and utilize various technology platforms in a range of geographies. Our reportable segments are organized as follows: The Communications segment provides wireless and wireline telecom and broadband services to consumers located in the United States and businesses globally. Our business strategies reflect integrated product offerings that cut across product lines and utilize shared assets. This segment contains the following business units: • Mobility provides nationwide wireless service and equipment. • Busine

In [65]:
import numpy as np, faiss

emb_unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)

d = emb_unit.shape[1]  # 768
index = faiss.IndexHNSWFlat(d, 32)
index.hnsw.efConstruction = 200
index.add(emb_unit)

# saving the faiss bse
faiss.write_index(index, "/content/drive/MyDrive/10k_processed/mpnet_hnsw.faiss")



In [66]:
import numpy as np, pandas as pd, faiss, os


os.makedirs("/content/drive/MyDrive/10k_processed", exist_ok=True)
## ID mapping along with FAISS INDEX fro future citation issue
ids = np.arange(len(emb_unit), dtype='int64')
base = faiss.IndexHNSWFlat(emb_unit.shape[1], 32)
base.hnsw.efConstruction = 200
index = faiss.IndexIDMap2(base)
index.add_with_ids(emb_unit, ids)

faiss.write_index(index, "/content/drive/MyDrive/10k_processed/mpnet_hnsw.faiss")

# Save metadata for citation
meta = pd.DataFrame([{
    "id": i,
    "doc_id": c.get("doc_id"),
    "filename": c.get("filename"),
    "page_start": c.get("page_start"),
    "page_end": c.get("page_end"),
    "text_clean": c.get("text_clean", "")
} for i, c in enumerate(chunks)])
meta.to_parquet("/content/drive/MyDrive/10k_processed/chunk_meta.parquet", index=False)


Query from faiss index mapper

In [67]:
# loading index, embedding and meta data for citaion
import faiss, pandas as pd
from sentence_transformers import SentenceTransformer

INDEX_PATH = "/content/drive/MyDrive/10k_processed/mpnet_hnsw.faiss"
META_PATH  = "/content/drive/MyDrive/10k_processed/chunk_meta.parquet"

index = faiss.read_index(INDEX_PATH)
meta  = pd.read_parquet(META_PATH)

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embedder.max_seq_length = 384

In [68]:
## Retriev from the index mapper along with citation

def retrieve_with_citations(query: str, k: int = 8):
    q = embedder.encode([query], normalize_embeddings=True)  # (1, 768)
    D, I = index.search(q, k)
    hits = []
    for rank, (lab, score) in enumerate(zip(I[0], D[0]), 1):
        row = meta.loc[meta["id"] == int(lab)].iloc[0]
        pages = (row.get("page_start"), row.get("page_end"))
        hits.append({
            "rank": rank,
            "score": float(score),
            "id": int(lab),
            "filename": row.get("filename"),
            "page_start": int(pages[0]) if pd.notna(pages[0]) else None,
            "page_end": int(pages[1]) if pd.notna(pages[1]) else None,
            "text": row.get("text_clean", "")
        })
    return hits

def build_context(hits, max_chunks=5):
    blocks = []
    for h in hits[:max_chunks]:
        if h["page_start"] and h["page_end"]:
            cite = f"[{h['filename']} pp.{h['page_start']}–{h['page_end']}]"
        else:
            cite = f"[{h['filename']}]"
        blocks.append(f"{cite}\n{h['text']}")
    return "\n\n".join(blocks)

In [69]:
# Loading the finetuned model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "iamAbhishek01/mistral7b-finance-merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.57G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [70]:
## Prompt Engineering for faiss quering
def answer_with_rag(question: str, k: int = 8, max_chunks: int = 5, max_new_tokens: int = 400):
    #retrieve
    hits = retrieve_with_citations(question, k=k)
    context = build_context(hits, max_chunks=max_chunks)

    #build prompt
    prompt = f"""You are a finance assistant. Answer strictly from the CONTEXT.
If the context is insufficient, say you don't know.
Always cite sources in brackets using filename and page range.

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=False
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    answer = text.split("ANSWER:")[-1].strip()
    return answer, hits

In [71]:
q = "What risk factors did Apple highlight for fiscal year 2024?"
answer, hits = answer_with_rag(q, k=12, max_chunks=5, max_new_tokens=350)
print("ANSWER:\n", answer, "\n")
print("CITATIONS:")
for h in hits[:5]:
    ps = f"pp.{h['page_start']}–{h['page_end']}" if h['page_start'] and h['page_end'] else "pages n/a"
    print(f"- {h['filename']} ({ps})  score={h['score']:.3f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


ANSWER:
 The Company’s financial position and growth are subject to risks related to global economic, political, and legal developments, as well as commercial and consumer risks such as acquisitions, dispositions, joint ventures, and share-based compensation. 

CITATIONS:
- apple_2024_10K.pdf (pp.7–8)  score=0.576
- General_Electric_10K.pdf (pp.38–39)  score=0.643
- apple_2024_10K.pdf (pp.24–24)  score=0.667
- apple_2024_10K.pdf (pp.50–51)  score=0.683
- apple_2024_10K.pdf (pp.27–28)  score=0.701


## Question from the documents

1.
Q: Since when has Alan R. Mulally served on the company’s Board of Directors?
A: Since July 2014.

2.
Q: What roles did Alan hold at Ford Motor Company, and during what years?
A: President and Chief Executive Officer from September 2006 through June 2014, and he was also a member of Ford’s board and served on its finance committee during the same period.

3.
Q: Which global automotive company did Alan lead as CEO?
A: Ford Motor Company.

4.
Q: Before Ford, what was Alan’s title at Boeing between March 2001 and September 2006?
A: Executive Vice President of The Boeing Company and President and Chief Executive Officer of Boeing Commercial Airplanes, Inc.

5.
Q: On which executive body at Boeing did Alan serve?
A: The Boeing Executive Council.

6.
Q: What Boeing division did Alan lead prior to his Commercial Airplanes role?
A: Boeing’s space and defense business (as President).

7.
Q: Name two government or advisory bodies Alan served on outside his corporate roles.
A: He co-chaired the Washington Competitiveness Council and sat on the U.S. Air Force Scientific Advisory Board.
(Other valid picks from the paragraph: advisory boards of NASA, University of Washington, University of Kansas, MIT.)

8.
Q: List three universities whose advisory boards included Alan.
A: University of Washington, University of Kansas, and the Massachusetts Institute of Technology.

9.
Q: Did Alan serve on Ford’s finance committee? If so, when?
A: Yes, from September 2006 through June 2014.

10.
Q: What time span did Alan serve as CEO of Boeing Commercial Airplanes?
A: From March 2001 to September 2006.

11. (multi-hop)
Q: Was Alan simultaneously on Ford’s board and serving as Ford’s CEO?
A: Yes. From September 2006 through June 2014 he was both on Ford’s board (and its finance committee) and served as President and CEO.

12. (negative control)
Q: Did the paragraph say Alan worked at General Motors?
A: No. It mentions Ford and Boeing, not General Motors.

In [72]:
qa_items = [
    ("Since when has Alan R. Mulally served on the company’s Board of Directors?", "since july 2014"),
    ("What roles did Alan hold at Ford Motor Company, and during what years?", "president and chief executive officer from september 2006 through june 2014"),
    ("Which global automotive company did Alan lead as CEO?", "ford motor company"),
    ("What was Alan’s title at Boeing between March 2001 and September 2006?", "executive vice president of the boeing company and president and chief executive officer of boeing commercial airplanes"),
    ("On which executive body at Boeing did Alan serve?", "boeing executive council"),
    ("What Boeing division did Alan lead prior to his Commercial Airplanes role?", "boeing’s space and defense business"),
    ("Name two government or advisory bodies Alan served on outside his corporate roles.", "washington competitiveness council; u.s. air force scientific advisory board"),
    ("List three universities whose advisory boards included Alan.", "university of washington; university of kansas; massachusetts institute of technology"),
    ("Did Alan serve on Ford’s finance committee? If so, when?", "yes; september 2006 through june 2014"),
    ("What time span did Alan serve as CEO of Boeing Commercial Airplanes?", "march 2001 to september 2006"),
    ("Was Alan simultaneously on Ford’s board and serving as Ford’s CEO?", "yes"),
    ("Did the paragraph say Alan worked at General Motors?", "no"),
]


def _norm(s: str) -> str:
    s = s or ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

results = []
for i, (q, gold) in enumerate(qa_items, 1):
    ans, hits = answer_with_rag(q, k=12, max_chunks=5, max_new_tokens=300)
    norm_ans  = _norm(ans)
    norm_gold = _norm(gold)
    ok = norm_gold in norm_ans
    top_cite = f"{hits[0]['filename']} pp.{hits[0]['page_start']}–{hits[0]['page_end']}" if hits else "n/a"
    results.append({
        "idx": i,
        "question": q,
        "expected_contains": gold,
        "answer": ans,
        "match": ok,
        "top_citation": top_cite,
        "top_score": hits[0]['score'] if hits else None
    })

df_res = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 180)
display(df_res)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,idx,question,expected_contains,answer,match,top_citation,top_score
0,1,Since when has Alan R. Mulally served on the company’s Board of Directors?,since july 2014,"Alan R. Mulally has served as a member of the Board of Directors since July 2014, and he has served as the Executive Vice President and Chief Financial Officer from March 1991 ...",True,AIG_10K.pdf pp.6–6,0.543645
1,2,"What roles did Alan hold at Ford Motor Company, and during what years?",president and chief executive officer from september 2006 through june 2014,"Alan served as President and Chief Executive Officer from September 2006 to June 2014, and as Executive Vice President from February 5, 2025 to February 5, 2024.",False,AIG_10K.pdf pp.6–6,0.796068
2,3,Which global automotive company did Alan lead as CEO?,ford motor company,"Alan R. Mulally served as CEO from July 2014 to September 2014, and as President and Chief Executive Officer from September 2006 to September 2014. He also served as co-chair o...",False,AIG_10K.pdf pp.6–6,0.728023
3,4,What was Alan’s title at Boeing between March 2001 and September 2006?,executive vice president of the boeing company and president and chief executive officer of boeing commercial airplanes,"The sum of each quarter’s per-share amount may not equal the total per-share amount for the respective year, and the sum of per-share amounts from continuing operations and dis...",False,AIG_10K.pdf pp.6–6,0.666628
4,5,On which executive body at Boeing did Alan serve?,boeing executive council,"Alan served as President, Server and Tools from 2011 to 2013, and Senior Vice President, Search, Portal, and Advertising from 2009 to 2011. He also served as Executive Vice Pre...",False,AIG_10K.pdf pp.6–6,0.643922
5,6,What Boeing division did Alan lead prior to his Commercial Airplanes role?,boeing’s space and defense business,[AIG_10K.pdf pp.10–10]\nAir Force Scientific Advisory Board. Paul S. Otellini • Global business leadership and extensive financial and management expertise as former President ...,False,AIG_10K.pdf pp.6–6,0.805707
6,7,Name two government or advisory bodies Alan served on outside his corporate roles.,washington competitiveness council; u.s. air force scientific advisory board,"Alan R. Mulally has served as an Executive Vice President since March 2017, and as Chief Accounting Officer since April 2019. He also served as co-chair of the Washington Compe...",False,AIG_10K.pdf pp.6–6,0.912262
7,8,List three universities whose advisory boards included Alan.,university of washington; university of kansas; massachusetts institute of technology,"The main function of the Audit Committee is to oversee accounting and financial reporting processes. This includes selecting and hiring independent auditors, developing financi...",False,AIG_10K.pdf pp.10–10,0.968042
8,9,"Did Alan serve on Ford’s finance committee? If so, when?",yes; september 2006 through june 2014,"The costco-chair of the Office of the Chair and Chief Executive Officer, and a member of the Sustainability, Innovation and Policy Committee of the Board of Directors.",False,AIG_10K.pdf pp.6–6,0.898487
9,10,What time span did Alan serve as CEO of Boeing Commercial Airplanes?,march 2001 to september 2006,"Alan served as President of Boeing Commercial Airplanes, Inc. and President and Chief Operating Officer of Ford Motor Company and President and Chief Operating Officer of Avio ...",False,AIG_10K.pdf pp.6–6,0.675211


## Evaluation Metrics :: RAGAS

In [88]:

## Questions
questions = [
    "What major risks does Alphabet cite from operating internationally and from financial exposures?",
    "Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.",
    "Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorships and executive experience, her education and credential, and Alan R. Mulally’s board service with dates",
    "In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",
    "Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for property and equipment.",
    "What macroeconomic and regulatory factors could materially impact Walmart, and how might they affect Walmart’s demand, margins, costs, inventory, suppliers, and partnerships?",
    "In brief, what were Pfizer’s key 2024 items (gains, dividends, charges) and what intangible-asset impairments did it record, including fair-value levels and valuation method?",
    "What does Ford’s 2024 Form 10-K cover and how is the company classified, including listing details?"

]
import pandas as pd

def ask_batch_to_df(
    questions,
    k=12,
    max_chunks=5,
    max_new_tokens=350,
):
    rows = []
    for q in questions:
        try:
            ans, _hits = answer_with_rag(q, k=k, max_chunks=max_chunks, max_new_tokens=max_new_tokens)
            ans = (ans or "").strip()
        except Exception as e:
            ans = f"[ERROR] {e}"
        rows.append({"question": q, "prediction": ans})
    return pd.DataFrame(rows)

# run
df_preds = ask_batch_to_df(questions, k=12, max_chunks=5, max_new_tokens=350)
df_preds



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,question,prediction
0,What major risks does Alphabet cite from operating internationally and from financial exposures?,Alphabet Inc.
1,"Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.","[aIG_10K.pdf pp.107–107]\nFORD MOTOR COMPANY BY: /s/ Mark Kosman Mark Kosman, Chief Accounting Officer (principal accounting officer) Date: February 5, 2025 Pursuant to the req..."
2,"Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorshi...","John has served as the President of Stanford University since September 2000, and he has also served as a member of the Board of Directors of Cisco Systems, Inc. and as the Cha..."
3,"In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",[ERROR] CUDA out of memory. Tried to allocate 618.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 52.12 MiB is free. Process 167004 has 14.69 GiB memory in use. Of the...
4,"Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for p...","Microsoft uses the market approach, which includes three levels of inputs and liabilities in active markets, and observable inputs for the asset or liability."
5,"What macroeconomic and regulatory factors could materially impact Walmart, and how might they affect Walmart’s demand, margins, costs, inventory, suppliers, and partnerships?","Walmart Inc. Consolidated Statements of Income Fiscal Years Ended January 31, (Dollar amounts and retail square feet in millions) 2025 2024 2023 Walmart Inc. Consolidated State..."
6,"In brief, what were Pfizer’s key 2024 items (gains, dividends, charges) and what intangible-asset impairments did it record, including fair-value levels and valuation method?",[ERROR] CUDA out of memory. Tried to allocate 3.02 GiB. GPU 0 has a total capacity of 14.74 GiB of which 2.46 GiB is free. Process 167004 has 12.28 GiB memory in use. Of the al...
7,"What does Ford’s 2024 Form 10-K cover and how is the company classified, including listing details?","Ford’s 2024 Form 10-K is $48,960,272,506,076 shares of Common Stock, and $38,497,761 shares of Total Stock."


In [89]:
ground_truth = [
    "Alphabet flags anti-bribery compliance (the FCPA and comparable local laws), cross-country labor and HR differences that add operational complexity, and foreign-currency risk that can depress revenue and earnings even with partial hedging (which itself introduces risk). It also notes exposure to changes in the fair value of its debt and equity investments due to liquidity, credit, market moves, interest rates, and regulation, and says several items—non-marketable securities, certain stock-based awards, and some acquisition assets/liabilities—require subjective fair-value estimates.",
    "Alan R. Mulally has served on the company’s Board of Directors since July 2014. He was President and CEO of Ford Motor Company from September 2006 to June 2014, while also serving on Ford’s board and its finance committee during that same period. Before Ford, from March 2001 to September 2006, he was Executive Vice President of The Boeing Company and President and CEO of Boeing Commercial Airplanes, and he sat on the Boeing Executive Council. Earlier, he served as President of Boeing’s space and defense business. Beyond those roles, he was co-chair of the Washington Competitiveness Council and served on advisory boards for NASA, the University of Washington, the University of Kansas, MIT, and the U.S. Air Force Scientific Advisory Board.",
    "John announced he will resign as President of Stanford University in August 2016. Ann Mather has been on our Board since November 2005 and currently serves on the boards of Arista Networks (since June 2013, audit committee chair), Glu Mobile (since September 2005, nominating and corporate governance committee), Netflix (since July 2010, audit committee chair), and Shutterfly (since May 2013, audit committee). She has been an independent trustee of the Dodge & Cox Funds since May 2011. Previously, she was a director of MoneyGram International (May 2010–May 2013) and Solazyme (April 2011–November 2014). Her executive roles include EVP & CFO at Pixar (1999–2004) and earlier EVP & CFO at Village Roadshow Pictures. She holds an M.A. from Cambridge University and is a chartered accountant. Alan R. Mulally has served on our Board since July 2014.",
    "Mostly leased warehouses and sales offices; about 80% owned for testing/prototype/operations space. Outside the U.S., Ford owns most plants and engineering centers, while many parts distribution centers are leased or vendor-provided. Overall it uses 375+ facilities in 24 countries, including 41 manufacturing/assembly plants supporting Ford Blue, Model e, and Ford Pro. Ford’s lone consolidated manufacturing JV is Ford Vietnam Limited (Ford 75% / VEAM affiliate 25%), which assembles and distributes Ford vehicles",
    "Level 2 uses observable inputs (e.g., agency and municipal securities, corporates, MBS/ABS, foreign gov bonds; plus cleared and OTC derivatives) priced with market data. Level 3 relies on unobservable inputs and models (DCF, option pricing) for items like certain corporates/munis and impaired goodwill/intangibles. Equity investments without readily determinable fair values are measured nonrecurringly using the best available techniques (market quotes/comparables or DCF). Other current financial assets/liabilities approximate carrying value. Property and equipment is at cost, depreciated straight-line over the shorter of useful life or lease term.",
    "Higher rates and energy costs, inflation/deflation, weak housing, unemployment, lower GDP/disposable income, tight credit, FX swings, tax and healthcare law changes, tariffs/trade barriers, or recession can curb demand, push mix to lower-margin items, slow inventory turns and force markdowns. They also lift COGS and SG&A (transport, labor, insurance, healthcare, commodities), risk impairments, strain suppliers (higher costs or reduced output), and cause partnerships/alliances to underperform.",
    "In 2024 Pfizer booked $945M gains from partial Haleon sales, $272M dividends from ViiV, and a $420M charge tied to an expected facility sale from discontinuing the DMD program. It recorded $3.295B of intangible impairments (all Level 3 fair value): IPR&D $1.873B, developed tech rights $943M, finite-lived brand $475M, and licensing $5M; fair value was measured via the income approach (multi-period excess earnings/DCF). (FYI context: 2023 included ViiV $265M, Nimbus $211M dividends and a $222M gain on an early-stage gene therapy divestiture to Alexion; 2022 included ViiV $314M, TSAs net $142M, and $77M contingent-consideration charges.",
    "Ford filed a Form 10-K for the year ended Dec 31, 2024 (Commission File 1-3950). It’s a Delaware registrant (EIN 38-0549190) headquartered at One American Road, Dearborn, MI 48126 (313-322-3000). Ford is a well-known seasoned issuer and a large accelerated filer (not smaller or emerging growth). Listed on the NYSE: Common Stock (ticker F) and senior notes FPRB, FPRC, FPRD."

]

In [90]:

assert len(ground_truth) == len(df_preds), "ground_truth length must match df_preds"


df_preds["ground_truth"] = ground_truth


df_preds["ground_truths"] = [
    [gt] if gt is not None and gt != "" else []
    for gt in ground_truth
]

df_preds.head()


,question,prediction,ground_truth,ground_truths
0,What major risks does Alphabet cite from operating internationally and from financial exposures?,Alphabet Inc.,"Alphabet flags anti-bribery compliance (the FCPA and comparable local laws), cross-country labor and HR differences that add operational complexity, and foreign-currency risk t...","[Alphabet flags anti-bribery compliance (the FCPA and comparable local laws), cross-country labor and HR differences that add operational complexity, and foreign-currency risk ..."
1,"Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.","[aIG_10K.pdf pp.107–107]\nFORD MOTOR COMPANY BY: /s/ Mark Kosman Mark Kosman, Chief Accounting Officer (principal accounting officer) Date: February 5, 2025 Pursuant to the req...","Alan R. Mulally has served on the company’s Board of Directors since July 2014. He was President and CEO of Ford Motor Company from September 2006 to June 2014, while also serv...","[Alan R. Mulally has served on the company’s Board of Directors since July 2014. He was President and CEO of Ford Motor Company from September 2006 to June 2014, while also ser..."
2,"Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorshi...","John has served as the President of Stanford University since September 2000, and he has also served as a member of the Board of Directors of Cisco Systems, Inc. and as the Cha...",John announced he will resign as President of Stanford University in August 2016. Ann Mather has been on our Board since November 2005 and currently serves on the boards of Ari...,[John announced he will resign as President of Stanford University in August 2016. Ann Mather has been on our Board since November 2005 and currently serves on the boards of Ar...
3,"In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",[ERROR] CUDA out of memory. Tried to allocate 618.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 52.12 MiB is free. Process 167004 has 14.69 GiB memory in use. Of the...,"Mostly leased warehouses and sales offices; about 80% owned for testing/prototype/operations space. Outside the U.S., Ford owns most plants and engineering centers, while many ...","[Mostly leased warehouses and sales offices; about 80% owned for testing/prototype/operations space. Outside the U.S., Ford owns most plants and engineering centers, while many..."
4,"Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for p...","Microsoft uses the market approach, which includes three levels of inputs and liabilities in active markets, and observable inputs for the asset or liability.","Level 2 uses observable inputs (e.g., agency and municipal securities, corporates, MBS/ABS, foreign gov bonds; plus cleared and OTC derivatives) priced with market data. Level ...","[Level 2 uses observable inputs (e.g., agency and municipal securities, corporates, MBS/ABS, foreign gov bonds; plus cleared and OTC derivatives) priced with market data. Level..."


In [91]:
df_ragas = df_preds.copy()
df_ragas = df_ragas.rename(columns={"prediction": "answer"})

In [92]:
def get_contexts(q, k=5):
    hits = retrieve_with_citations(q, k=k)
    return [h["text"] for h in hits]

df_ragas["contexts"] = df_ragas["question"].apply(lambda q: get_contexts(q, k=5))

df_ragas = df_ragas[["question", "answer", "contexts", "ground_truths"]]

df_ragas.head()


,question,answer,contexts,ground_truths
0,What major risks does Alphabet cite from operating internationally and from financial exposures?,Alphabet Inc.,"[In addition, our products and services are highly technical and complex and have contained in the past, and may contain in the future, errors or vulnerabilities, which could r...","[Alphabet flags anti-bribery compliance (the FCPA and comparable local laws), cross-country labor and HR differences that add operational complexity, and foreign-currency risk ..."
1,"Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.","[aIG_10K.pdf pp.107–107]\nFORD MOTOR COMPANY BY: /s/ Mark Kosman Mark Kosman, Chief Accounting Officer (principal accounting officer) Date: February 5, 2025 Pursuant to the req...","[Alan R. Mulally has served as a member of our Board of Directors since July 2014. Alan served as President and Chief Executive Officer of Ford Motor Company, a global automoti...","[Alan R. Mulally has served on the company’s Board of Directors since July 2014. He was President and CEO of Ford Motor Company from September 2006 to June 2014, while also ser..."
2,"Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorshi...","John has served as the President of Stanford University since September 2000, and he has also served as a member of the Board of Directors of Cisco Systems, Inc. and as the Cha...",[John has announced that he plans to resign from his position as the President of Stanford University in August 2016. Ann Mather has served as a member of our Board of Director...,[John announced he will resign as President of Stanford University in August 2016. Ann Mather has been on our Board since November 2005 and currently serves on the boards of Ar...
3,"In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",[ERROR] CUDA out of memory. Tried to allocate 618.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 52.12 MiB is free. Process 167004 has 14.69 GiB memory in use. Of the...,"[The majority of the warehouses that we operate are leased, although many of our manufacturing and assembly facilities contain some warehousing space. Substantially all of our ...","[Mostly leased warehouses and sales offices; about 80% owned for testing/prototype/operations space. Outside the U.S., Ford owns most plants and engineering centers, while many..."
4,"Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for p...","Microsoft uses the market approach, which includes three levels of inputs and liabilities in active markets, and observable inputs for the asset or liability.","[Fair value is defined as the exit price, or the amount that would be received to sell an asset or paid to transfer a liability in an orderly transaction between market partici...","[Level 2 uses observable inputs (e.g., agency and municipal securities, corporates, MBS/ABS, foreign gov bonds; plus cleared and OTC derivatives) priced with market data. Level..."


Loaded 8 rows; examples of contexts lens: [5, 5, 5, 5, 5]


In [6]:
## Due to complex implimentaion i have referenced this ccode from ChatGPT

import os, re, math, numpy as np, pandas as pd
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from openai import OpenAI


os.environ["OPENAI_API_KEY"] = "xxxxxxxxxxxxxxxxxxx"
JUDGE_MODEL = "gpt-4o"
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
TOP_K_SENTENCES = 6
SIM_THRESHOLD = 0.35


client = OpenAI()
embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 384

def _norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def _sentences(text: str) -> List[str]:

    parts = re.split(r"(?<=[.!?])\s+", text or "")
    return [p.strip() for p in parts if p.strip()]

def _embed(texts: List[str]) -> np.ndarray:
    vecs = embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    return vecs

def _cos(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))

def judge_yes_no(prompt: str) -> float:
    """
    Calls the judge LLM. Returns 1.0 for 'yes', 0.0 for 'no' (or in-between if model outputs a probability).
    We map 'yes'->1, 'no'->0, else try to parse a numeric score.
    """
    msg = [{"role": "user", "content": prompt}]
    out = client.chat.completions.create(model=JUDGE_MODEL, messages=msg, temperature=0)
    text = out.choices[0].message.content.strip().lower()

    m = re.search(r"(\b0(\.\d+)?|\b1(\.0+)?|\b0\.\d+|\b\d{1,2}%|\b100%)", text)
    if m:
        s = m.group(0)
        if s.endswith("%"):
            return min(1.0, max(0.0, float(s[:-1]) / 100.0))
        try:
            return min(1.0, max(0.0, float(s)))
        except:
            pass
    if "yes" in text and "no" not in text:
        return 1.0
    if "no" in text and "yes" not in text:
        return 0.0

    return 0.5


def answer_relevancy_score(question: str, answer: str) -> float:
    """
    relavancy of the answers
    """
    prompt = f"""
You are grading answer relevance. Score between 0 and 1 where:
0 = off-topic, 1 = fully answers the question.

Question:
{question}

Answer:
{answer}

Return only a number between 0 and 1 (or a percentage).
"""
    return judge_yes_no(prompt)

def faithfulness_score(answer: str, contexts: List[str]) -> float:
    """
    faithufullness score
    """
    ctx = "\n\n".join(contexts)
    prompt = f"""
You are grading faithfulness to context. Score between 0 and 1 where:
1 = every factual claim in the answer is supported by the CONTEXT, 0 = unsupported/hallucinated.

CONTEXT:
{ctx}

ANSWER:
{answer}

Return only a number between 0 and 1 (or a percentage).
"""
    return judge_yes_no(prompt)

def context_precision_score(question: str, contexts: List[str]) -> float:

    qv = _embed([question])[0]
    sent_list = []
    for c in contexts:
        sents = _sentences(c)[:TOP_K_SENTENCES]
        sent_list.extend(sents)
    if not sent_list:
        return 0.0
    sv = _embed(sent_list)
    sims = sv @ qv
    relevant = (sims >= SIM_THRESHOLD).sum()
    return float(relevant) / len(sent_list)

def context_recall_score(question: str, contexts: List[str], ground_truths: List[str]) -> float:

    gts = [g for g in ground_truths if g and g.strip()]
    if not gts:
        return 0.0

    sent_list = []
    for c in contexts:
        sent_list.extend(_sentences(c)[:TOP_K_SENTENCES])
    if not sent_list:
        return 0.0
    sent_vecs = _embed(sent_list)
    covered = 0
    for gt in gts:
        gt_vec = _embed([gt])[0]
        sims = sent_vecs @ gt_vec
        if float(np.max(sims)) >= SIM_THRESHOLD:
            covered += 1
    return covered / len(gts)


def evaluate_rag(df: pd.DataFrame) -> pd.DataFrame:
    """
    This will evaluate both predictiona and ground truth
    """
    rows = []
    for i, r in df.iterrows():
        q = _norm_text(r["question"])
        a = _norm_text(r["answer"])
        ctxs = list(r["contexts"])
        gts  = list(r["ground_truths"])

        ar  = answer_relevancy_score(q, a)
        fa  = faithfulness_score(a, ctxs)
        cp  = context_precision_score(q, ctxs)
        cr  = context_recall_score(q, ctxs, gts)

        rows.append({
            "question": q,
            "answer": a,
            "answer_relevancy": ar,
            "faithfulness": fa,
            "context_precision": cp,
            "context_recall": cr,
        })
    return pd.DataFrame(rows)


scores_df = evaluate_rag(df_ragas)
pd.set_option("display.max_colwidth", 180)
display(scores_df.head())

# Aggregate means
agg = scores_df[["answer_relevancy","faithfulness","context_precision","context_recall"]].mean().to_dict()
print("\n== Aggregate (mean) ==")
for k, v in agg.items():
    print(f"{k}: {v:.3f}")



,question,answer,answer_relevancy,faithfulness,context_precision,context_recall
0,What major risks does Alphabet cite from operating internationally and from financial exposures?,Alphabet Inc.,0.0,0.00,0.566667,1.0
1,"Summarize Alan R. Mulally’s service on the company’s board and his prior leadership at Ford and Boeing, including dates, committee memberships, and key advisory roles.","[aIG_10K.pdf pp.107–107] FORD MOTOR COMPANY BY: /s/ Mark Kosman Mark Kosman, Chief Accounting Officer (principal accounting officer) Date: February 5, 2025 Pursuant to the requ...",0.0,0.00,0.600000,1.0
2,"Summarize the key governance and career details mentioned: John’s planned resignation from Stanford, Ann Mather’s current board seats and committee roles, her prior directorshi...","John has served as the President of Stanford University since September 2000, and he has also served as a member of the Board of Directors of Cisco Systems, Inc. and as the Cha...",0.0,0.75,0.666667,1.0
3,"In brief, what does Ford’s global facilities footprint and ownership/lease mix look like as of Dec 31, 2024?",[ERROR] CUDA out of memory. Tried to allocate 618.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 52.12 MiB is free. Process 167004 has 14.69 GiB memory in use. Of the...,0.0,0.00,0.633333,1.0
4,"Briefly explain how Microsoft measures fair value for Level 2 and Level 3 items, how it values equity investments without readily determinable fair values, and its policy for p...","Microsoft uses the market approach, which includes three levels of inputs and liabilities in active markets, and observable inputs for the asset or liability.",0.5,0.00,0.766667,1.0



== Aggregate (mean) ==
answer_relevancy: 0.062
faithfulness: 0.094
context_precision: 0.633
context_recall: 1.000
